# Tutorial: The Python Data Science Ecosystem

> **Type:** Tutorial. Follow along step by step. Your only job is to make each cell run.  
> **Time:** ~2 hours  
> **Prerequisites:** Python basics (lists, dicts, functions, pip)  
> **What you'll build:** A working familiarity with every major library in the stack (numpy, pandas, matplotlib, seaborn, scipy, sklearn), by using each one on real data.

---

## What We're Building

This notebook is a guided tour of the Python data science toolkit. Each section covers one library. By the end, you'll have touched every tool you'll use for the rest of this course, and you'll know the *feel* of each one, even if you don't yet know all the details.

Don't worry about memorising syntax. That's what `reference.md` and `howto.md` are for.

## Step 0: Setup

In [ ]:
# Uncomment to install if needed
# !pip install numpy pandas matplotlib seaborn scipy scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_iris
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
np.random.seed(42)

print('All libraries loaded successfully.')
print(f'  numpy  {np.__version__}')
print(f'  pandas {pd.__version__}')

---

## Part 1: NumPy — Fast Arrays

NumPy is the foundation. Everything else in the stack is built on it.

### 1a. Creating Arrays

In [ ]:
# From a list
a = np.array([1, 2, 3, 4, 5])
print('1D array:', a)
print('dtype:', a.dtype)      # NumPy inferred int64
print('shape:', a.shape)      # (5,) — one dimension, 5 elements

# 2D array (matrix)
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print('\n2D array shape:', M.shape)  # (2, 3) — 2 rows, 3 columns

# Useful constructors
print('\nzeros:', np.zeros(5))
print('ones:', np.ones((2, 3)))
print('range:', np.arange(0, 10, 2))   # 0, 2, 4, 6, 8
print('linspace:', np.linspace(0, 1, 5))  # 5 evenly spaced values 0→1
print('random:', np.random.randn(4).round(3))  # standard normal

### 1b. Why NumPy is Fast — Timing the Speedup Directly

In [ ]:
import timeit

N = 1_000_000  # one million elements
py_list = list(range(N))
np_array = np.arange(N, dtype=np.float64)

t_py = timeit.timeit(
    '[x * 2 for x in py_list]', globals=globals(), number=5
) / 5
t_np = timeit.timeit(
    'np_array * 2', globals=globals(), number=5
) / 5
print(f'Python loop: {t_py*1000:.2f} ms')
print(f'NumPy:       {t_np*1000:.3f} ms')
print(f'Speedup:     {t_py/t_np:.0f}x')

> **What just happened?** NumPy is typically 50–200x faster than a Python loop on the same operation, and the gap grows with array size. That's why you should *never* loop over array elements when a vectorised operation exists.

### 1c. Broadcasting and Boolean Indexing

In [ ]:
scores = np.array([72, 85, 61, 90, 55, 78, 92, 44])

# Broadcasting: operate on every element without a loop
scores_scaled = (scores - scores.mean()) / scores.std()
print('Original:', scores)
print('Scaled (mean=0, std=1):', scores_scaled.round(2))

# Boolean mask: select elements that meet a condition
passing = scores[scores >= 70]
print(f'\nPassing scores (>=70): {passing}')
print(f'Pass rate: {(scores >= 70).mean():.0%}')

# Aggregations
print(f'\nMean: {scores.mean():.1f}')
print(f'Std:  {scores.std():.1f}')
print(f'Min:  {scores.min()}, Max: {scores.max()}')

---

## Part 2: pandas — Labelled Tabular Data

### 2a. Loading a Dataset

In [ ]:
# seaborn bundles several classic datasets for practice
df = sns.load_dataset('titanic')

print(f'Shape: {df.shape}  ({df.shape[0]} passengers, {df.shape[1]} columns)')
df.head()

In [ ]:
# Always run these three after loading
df.info()          # column types and non-null counts

In [ ]:
df.describe()      # summary statistics

### 2b. Selecting, Filtering, Sorting

In [ ]:
# Select one column → Series
ages = df['age']
print('Type:', type(ages))
print('Mean age:', ages.mean().round(1))

# Select multiple columns → DataFrame
subset = df[['survived', 'pclass', 'sex', 'age']]
subset.head()

In [ ]:
# Filter rows: first class survivors
first_class_survivors = df[(df['pclass'] == 1) & (df['survived'] == 1)]
print(f'First class survivors: {len(first_class_survivors)}')

# Sort by fare descending (seaborn's titanic has no 'name' column,
# unlike the classic Kaggle CSV -- use sex/class/embark_town instead)
df.sort_values('fare', ascending=False)[['sex', 'pclass', 'embark_town', 'fare']].head()

### 2c. Grouping and Aggregating

In [ ]:
# Survival rate by passenger class
survival_by_class = df.groupby('pclass')['survived'].agg(['mean', 'count'])
survival_by_class.columns = ['survival_rate', 'count']
survival_by_class['survival_rate'] = survival_by_class['survival_rate'].round(2)
print(survival_by_class)

In [ ]:
# Pivot table: survival rate by class AND sex
pivot = df.pivot_table(values='survived', index='pclass', columns='sex', aggfunc='mean')
pivot.round(2)

> **What just happened?** In 4 lines you computed a meaningful business metric across several dimensions. This is pandas at its best: expressive aggregations on labelled data.

---

## Part 3: matplotlib — Three Chart Types

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Line chart ---
x = np.linspace(0, 4 * np.pi, 200)
axes[0].plot(x, np.sin(x), color='#4E79A7', linewidth=2, label='sin(x)')
axes[0].plot(x, np.cos(x), color='#E15759', linewidth=2, label='cos(x)')
axes[0].set_title('Line Chart')
axes[0].set_xlabel('x')
axes[0].legend()

# --- Bar chart ---
classes = ['1st', '2nd', '3rd']
rates   = df.groupby('pclass')['survived'].mean().values
axes[1].bar(classes, rates, color=['#59A14F', '#F28E2B', '#E15759'], edgecolor='white')
axes[1].set_title('Bar Chart: Survival Rate by Class')
axes[1].set_ylabel('Survival Rate')
axes[1].set_ylim(0, 1)
for i, v in enumerate(rates):
    axes[1].text(i, v + 0.02, f'{v:.0%}', ha='center', fontweight='bold')

# --- Scatter chart ---
sample = df.dropna(subset=['age', 'fare']).sample(200, random_state=42)
scatter = axes[2].scatter(
    sample['age'], sample['fare'],
    c=sample['survived'], cmap='RdYlGn',
    alpha=0.6, s=40, edgecolors='none'
)
axes[2].set_title('Scatter: Age vs Fare (green=survived)')
axes[2].set_xlabel('Age')
axes[2].set_ylabel('Fare')
plt.colorbar(scatter, ax=axes[2])

plt.tight_layout()
plt.savefig('matplotlib_three_charts.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Three basic matplotlib charts. Line chart: sin(x) and cos(x) from 0 to 4π, two offset waves between −1 and 1. Bar chart: Titanic survival rate by class, 63% in first, 47% in second and 24% in third. Scatter: fare against age for 200 passengers, coloured by survival. Most fares are under 100. The few high fares are mostly survivors.

---

## Part 4: seaborn — Statistical Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution plot: age by survival status
sns.histplot(
    data=df, x='age', hue='survived',
    kde=True, bins=25, ax=axes[0],
    palette={0: '#E15759', 1: '#59A14F'}
)
axes[0].set_title('Age Distribution\nby Survival')

# Correlation heatmap
numeric_cols = ['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], square=True, linewidths=0.5)
axes[1].set_title('Correlation Heatmap')

# Categorical comparison
sns.barplot(data=df, x='pclass', y='survived', hue='sex',
            ax=axes[2], palette={'male': '#4E79A7', 'female': '#F28E2B'})
axes[2].set_title('Survival Rate\nby Class and Sex')
axes[2].set_ylabel('Survival Rate')

plt.tight_layout()
plt.savefig('seaborn_charts.png', dpi=120, bbox_inches='tight')
plt.show()

**Chart description.** Three seaborn charts of the Titanic data. Left: age histograms for survivors and non-survivors. Both peak in the twenties. Young children survive more often than not. Middle: a correlation heatmap. Survival correlates with class at −0.34 and with fare at 0.26. Class and fare correlate at −0.55. Right: survival rate by class and sex, with error bars. Women survive at about 97%, 92% and 50% in first, second and third class. Men survive at about 37%, 16% and 14%.

In [ ]:
# Pairplot: shows all pairwise relationships at once
# (Great for initial exploration of a dataset)
pair_cols = ['age', 'fare', 'pclass', 'survived']
pair_data = df[pair_cols].dropna()

g = sns.pairplot(pair_data, hue='survived',
                 palette={0: '#E15759', 1: '#59A14F'},
                 plot_kws={'alpha': 0.4})
g.fig.suptitle('Pairplot: Titanic Features', y=1.02)
plt.savefig('seaborn_pairplot.png', dpi=100, bbox_inches='tight')
plt.show()

**Chart description.** A pairplot of age, fare and class, coloured by survival. Scatter plots fill the off-diagonal panels and density curves sit on the diagonal. Most fares are low, with a long tail up to about 500. Third class has far more non-survivors than survivors. First class has more survivors.

> **What just happened?** The pairplot produced 9 charts from one function call: a 3 × 3 grid with scatter plots for every pair of age, fare and class, density curves on the diagonal, and colours by survival. `survived` became the colour (`hue`), so it isn't one of the axes. That's seaborn's value: statistical insight with very little code.

---

## Part 5: scipy — A Statistical Test

In [ ]:
# Question: Did survivors pay significantly higher fares?
survivors     = df[df['survived'] == 1]['fare'].dropna()
non_survivors = df[df['survived'] == 0]['fare'].dropna()

print(f'Survivor mean fare:     £{survivors.mean():.2f}')
print(f'Non-survivor mean fare: £{non_survivors.mean():.2f}')

# Independent samples t-test
t_stat, p_value = stats.ttest_ind(survivors, non_survivors)

print(f'\nt-statistic: {t_stat:.3f}')
print(f'p-value:     {p_value:.2e}')

if p_value < 0.05:
    print('\nConclusion: The difference is statistically significant (p < 0.05).')
    print('Survivors paid materially higher fares on average.')
else:
    print('\nConclusion: No significant difference detected.')

> **What just happened?** `scipy.stats.ttest_ind` ran a two-sample t-test in one line. The p-value tells us how likely a fare difference this large would be if it were down to chance alone. You'll use `scipy.stats` throughout the curriculum for hypothesis tests like this.

---

## Part 6: scikit-learn — The ML API Pattern

We won't explain *how* k-Nearest Neighbours works yet (that's Module 11). The goal here is to see the **pattern**. Every sklearn model follows the same three-step API.

In [ ]:
# Load the iris dataset: 150 flowers, 4 features, 3 species
iris = load_iris(as_frame=True)
X = iris.data      # features (sepal/petal measurements)
y = iris.target    # labels (0=setosa, 1=versicolor, 2=virginica)

print(f'Features shape: {X.shape}')  # (150, 4)
print(f'Target shape:   {y.shape}')  # (150,)
print(f'Classes: {iris.target_names}')
X.head()

In [ ]:
# Step 1: Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Training samples: {len(X_train)}')
print(f'Test samples:     {len(X_test)}')

In [ ]:
# Step 2: Create the model
model = KNeighborsClassifier(n_neighbors=5)

# Step 3: FIT: learn from training data
model.fit(X_train, y_train)

# Step 4: PREDICT: apply to test data
y_pred = model.predict(X_test)

# Step 5: EVALUATE
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.0%}')

# Show a few predictions vs actuals
comparison = pd.DataFrame({
    'actual':    [iris.target_names[i] for i in y_test[:10]],
    'predicted': [iris.target_names[i] for i in y_pred[:10]],
})
comparison['correct'] = comparison['actual'] == comparison['predicted']
print('\nFirst 10 predictions:')
print(comparison.to_string(index=False))

> **What just happened?** You trained and evaluated a machine learning classifier in 5 lines of code. The algorithm (k-NN) doesn't matter yet. What matters is the pattern: `fit` → `predict` → `score`. It's the same for every sklearn estimator.

---

## Summary: One Page of Truth

In [ ]:
print('''
Library       When you reach for it
──────────────────────────────────────────────────────────────────
numpy         Array operations, math, linear algebra
pandas        Loading, cleaning, transforming, grouping tabular data
matplotlib    Custom, precise charts; anything seaborn can't do
seaborn       Statistical charts during EDA; fast, readable defaults
scipy.stats   Hypothesis tests, distributions, statistical functions
sklearn       ML models, preprocessing, cross-validation, metrics
──────────────────────────────────────────────────────────────────
''')

## What You Built

A working tour of the whole Python data science stack, from raw array operations to a trained ML classifier, using real datasets. You now have muscle memory for the basic syntax of every library you'll use in this course.

## What's Next

- Read `explanation.md` if you haven't. It explains *why* the stack is designed the way it is.
- Keep `reference.md` open as a cheat sheet for all six libraries.
- `howto.md` has recipes for the operations you'll repeat most often.
- **Extension challenge:** swap `KNeighborsClassifier` for `sklearn.tree.DecisionTreeClassifier`. It's the same fit/predict/score pattern with a different algorithm. Does accuracy change?

---

**Next module:** `07_data_cleaning_wrangling`, where you learn to deal with real, messy data before you can model it.